## Measurements
* Every observable trait has an associated operator $\hat{A}$.

* The only values that can be measured are the eigenvalues of eigenfunctions of $\hat{A}$

* For operator $\hat{A}$, functions of the form $\hat{A} f_a = a f_a$

* For a state $\psi$, the chance of seeing eigenvalue $a$ are:
  * $P(a) = |\int f_a \psi dx)|^2$

## Functions



In [10]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import sys, os
from ipywidgets import interact
from IPython.display import display, Math
from matplotlib.gridspec import GridSpec

In [11]:
def pib(x, l, n):
  box = (x > 0)
  box *= (x < l)
  return (2/l)**.5 * np.sin(n * np.pi * x / l)*box

def pibE(l, mass, n):
  h = 6.626e-34
  return n**2 * h**2 / (8*mass*l**2)


def pib2D(lmax, lx, ly, nx, ny,pixels=300):
  x = np.linspace(0, lmax, pixels)
  y = np.linspace(0, lmax, pixels)
  X, Y = np.meshgrid(x, y)
  psi = pib(X, lx, nx) * pib(Y, ly, ny)
  return X, Y, psi

def pib_p(x, lx, n, direction):
  box = (x>0)
  box *= (x<lx)
  return np.exp(direction *1j * n * np.pi * x/lx)*box

def lvl_pairs(n_max):
  sums = []
  pairs_ = []
  for n in range(1, n_max + 1):
    for m in range(1,n_max+1):
      sums.append(n+m)
      pairs_.append([n,m])
  pairs_ = np.array(pairs_)
  order = list(np.argsort(sums))
  return pairs_[order]

def energy_dict(lx, ly, mass, n_max):
  energy_dict = {}
  pairs = lvl_pairs(n_max)
  E_mag = [n**2/lx**2 + m**2/ly**2 for n,m in pairs ]
  order = np.argsort(E_mag)
  for n,m in pairs[order]:
    name = f'({n},{m})'
    energy_dict[name] = {}
    energy_dict[name]['E'] = pibE(lx, mass, n) + pibE(ly, mass, m)
    energy_dict[name]['nx'] = n
    energy_dict[name]['ny'] = m

  return energy_dict


In [12]:
def xy_operate(nmax=4, measurements=500):
  lx = 1e-9
  me = 9.101e-31
  ns = [n for n in range(1, nmax+1)]
  seed = np.random.randint(1, 100)

  @interact(
      nx = widgets.Dropdown(options=ns),
      ny = widgets.Dropdown(options=ns),
      n_measure = widgets.IntSlider(min=1,
                                    max=measurements,
                                    value=1,
                                    step=1,
                                    description="# Measurements"))
  def g(nx, ny, n_measure):


    fig, ax = plt.subplots(ncols=3, figsize=(10,3.))
    fig.subplots_adjust(wspace=0.4)
    npix = 300
    X, Y, psi = pib2D(1e-9, lx, lx, nx, ny,pixels=npix)

    x = np.linspace(0, lx, npix)

    px = pib(x, lx, nx)**2 / (pib(x, lx, nx)**2).sum()
    py = pib(x, lx, ny)**2 / (pib(x, lx, ny)**2).sum()
    np.random.seed(seed)
    x_list= np.random.choice(x, size=measurements+1,p=px)
    y_list= np.random.choice(x, size=measurements+1,p=py)

    ax[2].plot(x_list[:n_measure]*1e9,
               y_list[:n_measure]*1e9, 'k.',
               label="All measurements")
    ax[2].plot(x_list[n_measure]*1e9,
               y_list[n_measure]*1e9, 'r.',
               label='Last measurement')

    if n_measure > 0:
      xmean = np.mean(x_list[:n_measure]*1e9)
      xvariance = np.var(x_list[:n_measure]*1e9)
      ymean = np.mean(y_list[:n_measure]*1e9)
      yvariance = np.var(y_list[:n_measure]*1e9)
      ax[2].set_title(f"Avg. x:{xmean:.3f} nm, $\\sigma_x$={xvariance**.5:.2f}\n Avg. y: {ymean:.3f}, $\\sigma_y$={yvariance**.5:.2f}")
    else:
      ax[2].set_title(f"")

    ax[2].set_xlim(0, lx*1e9)
    ax[2].set_ylim(0, lx*1e9)
    ax[2].set_xlabel('x (nm)')
    ax[2].set_ylabel('y (nm)')
    ax[2].set_aspect('equal', adjustable='box')

    vmax = np.max(psi)
    vmin = -vmax
    im0 = ax[0].imshow(psi, vmin=vmin, vmax=vmax, cmap='seismic', extent=(0,1,0,1))
    ax[0].set_ylabel(f"$\\psi_{nx,ny}$")
    ax[0].set_xlabel("x (nm)")
    ax[0].set_ylabel("y (nm)")
    ax[0].set_title(f"$\\psi_{{ {nx},{ny}}}$")
    cbar = fig.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)

    ax[1].set_xlabel("x (nm)")
    ax[1].set_ylabel("y (nm)")
    im1 = ax[1].imshow(psi * psi, cmap='binary', extent=(0,1,0,1))
    ax[1].set_title(f"$\\psi^*_{{ {nx},{ny}}}\\psi^{{}}_{{ {nx},{ny}}}$")
    cbar = fig.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.04)

In [19]:

# Mixing two levels
def pib_mix(nmax, measurements):
  lx_m = 1e-9
  ly_m = 1e-9
  me = 9.101e-31
  e_dict = energy_dict(lx_m, ly_m, me, nmax)
  levels = [e for e in e_dict]
  seed = np.random.randint(1, 100)
  lmax = 1e-9

  @interact(
        state1=widgets.Dropdown(options=levels,
                                value=levels[0],
                                description='ψ1: (nx, ny)'),
        state2=widgets.Dropdown(options=levels,
                                value=levels[1],
                                description='ψ2: (nx, ny)'),
        theta=widgets.FloatSlider(
            value=np.pi/4,
            min=0,
            max=np.pi/2,
            step=0.05,
            description='θ'),
        n_measure = widgets.IntSlider(min=1,
                                    max=measurements,
                                    value=1,
                                    step=1,
                                    description="# Measurements"))
  def g(state1, state2, theta, n_measure):
    nx1 = e_dict[state1]['nx']
    ny1 = e_dict[state1]['ny']
    nx2 = e_dict[state2]['nx']
    ny2 = e_dict[state2]['ny']
    c1 = np.cos(theta)
    c2 = np.sin(theta)

    #fig, ax = plt.subplots(ncols=4, figsize=(10, 5))
    #fig.subplots_adjust(wspace=0.4)
    fig = plt.figure(figsize=(10,3))
    gs = GridSpec(1, 4, figure=fig, wspace=.5)
    ax = []
    ax.append(fig.add_subplot(gs[0]))
    ax.append(fig.add_subplot(gs[1]))
    ax.append(fig.add_subplot(gs[2]))
    ax.append(fig.add_subplot(gs[3]))

    X, Y, psi1 = pib2D(lmax, lx_m, ly_m, nx1, ny1)
    X, Y, psi2 = pib2D(lmax, lx_m, ly_m, nx2, ny2)
    psi = c1*psi1 + c2*psi2
    vmax = np.max(psi)
    vmin = -vmax

    ax[0].imshow(psi1, cmap='seismic',
                vmin=vmin, vmax=vmax,
                extent=(0, 1, 0, 1))
    ax[0].set_title(f'$\\psi_1$: $n_x$={nx1}, $n_y$={ny1}')

    ax[1].imshow(psi2, cmap='seismic',
                vmin=vmin, vmax=vmax,
                extent=(0, 1, 0, 1))
    ax[1].set_title(f'$\\psi_2$: $n_x$={nx2}, $n_y$={ny2}')
    ax[2].imshow(psi, cmap='seismic',vmin=vmin, vmax=vmax,
                extent=(0, 1, 0, 1))
    ax[2].set_title(
        f'$\\psi$: {c1:.2f}$\\psi_1$ + {c2:.2f}$\\psi_2$')
    E1 = e_dict[state1]['E']
    E2 = e_dict[state2]['E']

    
    np.random.seed(seed)
    Es = [E1, E2]
    Ps = [c1**2,c2**2]
    E_list= np.random.choice(Es, size=measurements+1,p=Ps)
    bins = np.linspace(np.min(E_list)*1.3**-1, np.max(E_list)*1.3, 100)
    hi = ax[3].hist(E_list[:n_measure], bins)
    ax[3].plot(E_list[n_measure], 1, 'r.')
    ax[3].set_ylim(0, measurements)
    ax[3].set_xlabel("E (J)")
    ax[3].set_ylabel("Counts")
    ax[3].text(E1, n_measure*c1**2 , f"$E_{{{nx1},{ny1}}}$", ha='left')
    ax[3].text(E2, n_measure*c2**2 , f"$E_{{{nx2},{ny2}}}$", ha='right')
    E_avg = np.mean(E_list[:n_measure])*1e19
    E_std = np.std(E_list[:n_measure])*1e19
    if n_measure > 0:
        ax[3].set_title(f"Avg. E =  {E_avg:.2f}$\\times10^{{-19}}$ J \n $\\sigma_E$ = {E_std:.2f} $\\times10^{{-19}}$ J")

    display(Math(rf"\psi = \cos(\theta)\psi_{{{nx1},{ny1}}}+\sin(\theta)\psi_{{{nx2},{ny2}}}"))
    display(Math(rf"\psi = {c1:.2f}\psi_{{{nx1},{ny1}}}+{c2:.2f}\psi_{{{nx2},{ny2}}}"))
    display(Math(rf"\langle \hat{{H}} \rangle = \cos(\theta)^2 E_{{{nx1},{ny1}}} + \sin(\theta)^2 E_{{{nx2},{ny2}}}"))
    display(Math(rf"\langle \hat{{H}} \rangle = {c1**2:.2f} E_{{{nx1},{ny1}}} + {c2**2:.2f}E_{{{nx2},{ny2}}}"))
    display(Math(rf"E_{{{nx1},{ny1}}} = {E1*1e19:.2f}\times 10^{{-19}} J"))
    display(Math(rf"E_{{{nx2},{ny2}}} = {E2*1e19:.2f}\times 10^{{-19}} J"))

In [37]:

def p_measure(nmax, measurements=1000):
  ns = [n for n in range(1, nmax+1)]
  seed = np.random.randint(1, 100)
  measurements = 1000

  h = 6.62607015e-34
  hbar = h/(2*np.pi)


  @interact(
        l = widgets.FloatSlider(min=.1, max=2, value=1, description='length (nm)'),
        n = widgets.Dropdown(options=ns),
        n_measure = widgets.IntSlider(min=1,
                                      max=measurements,
                                      value=1,
                                      step=1,
                                      description="# Measurements"))
  def g(l, n, n_measure):

    N = 2**14
    lx = l*1e-9
    x = np.linspace(-100*lx, 100*lx, N)
    dx = x[1]-x[0]

    psi = pib(x, lx, n)
    psi_k = (dx/(2*np.pi)**.5)*np.fft.fft(psi)
    k = 2*np.pi * np.fft.fftfreq(N, d=dx)
    p = hbar * k

    psi_k = np.fft.fftshift(psi_k)
    k = np.fft.fftshift(k)
    p = np.fft.fftshift(p)

    dp = p[1] - p[0]
    phi_p = psi_k / (np.sum(np.abs(psi_k)**2) * dp)**.5

    fig, ax = plt.subplots(ncols=3, figsize=(15,3.5))
    fig.subplots_adjust(wspace=0.2 )
    ax[0].plot(x*1e9, psi)
    ax[0].set_xlim(-lx*1e9, 2*lx*1e9)
    ax[0].set_ylim(-np.max(psi)*1.1, np.max(psi)*1.1)
    ax[1].plot(p, np.abs(phi_p)**2)


    p1_inf =  np.pi * hbar / lx
    ax[1].set_xlim(-10*p1_inf, 10*p1_inf)

    pp = np.abs(phi_p)**2/ np.sum(np.abs(phi_p)**2)
    np.random.seed(seed)
    p_list= np.random.choice(p, size=measurements+1,p=pp)

    bins = np.linspace(-10*p1_inf, 10*p1_inf, 50)
    ax[2].hist(p_list[:n_measure], bins=bins)
    ax[2].set_xlim(-10*p1_inf, 10*p1_inf)

    ax[0].set_title(f"$\\psi_{n}(x)$")
    ax[0].set_xlabel('x (nm)')
    ax[0].set_ylabel(f'$\\psi_{n}(x)$')
    ax[0].set_xlim(-.5, 2.5)

    ax[1].set_title("Momentum $P(p)$")#
    ax[1].set_xlabel("$p$ (kg m/s)")
    ax[1].set_ylabel(r"$|\langle p|\psi\rangle|^2$")
    pspan = 5
    ax[1].set_xlim(-pspan*1e-24, pspan*1e-24)
    ax[2].set_xlim(-pspan*1e-24, pspan*1e-24)
    ax[2].plot(p_list[n_measure], 1, 'r.')
    avgp = np.mean(p_list[:n_measure])
    if n_measure > 0:
      sp = np.std(p_list[:n_measure])
    else:
      sp = 0
    ax[2].set_title(f"Avg p={np.mean(avgp*1e24):.2f}$\\times 10^{{-24}}$ kg m s$^{{-1}}$ \n $\\sigma_p$={sp*1e24:.2f}$\\times 10^{{-24}}$ kg m s$^{{-1}}$")





## Energy Operator, the Hamiltonian $\hat{H}$

The particle-in-a-box (PIB) wave functions are eigenfunctions of $\hat{H}$. For a 2D PIB:

$\hat{H}\psi_{n_x,n_y}(x,y) = E_{n_x,n_y}\psi$


$E_{n_x,n_y} = \dfrac{h^2 }{8 m}\left(\dfrac{n_x^2}{L_x^2} + \dfrac{n_y^2}{L_y^2}\right)$

We can make a new state by adding $\psi_{n_x,n_y}(x,y)$ together:

$\psi(x,y) = c_1\psi_{n_{x1},n_{y1}}(x,y) + c_2\psi_{n_{x2},n_{y2}}(x,y)$

As long as $c_1^2 + c_2^2$ $=1$

In [21]:
pib_mix(4, 100)

interactive(children=(Dropdown(description='ψ1: (nx, ny)', options=('(1,1)', '(1,2)', '(2,1)', '(2,2)', '(1,3)…

## Position operators: $\hat{x}$ and $\hat{y}$

The position operators $\hat{x}=x$ and $\hat{y}=y$ are related to measuring position for a 2D wavefunction.

* For positions operators, the eigenvalues are points in space.

* The eigenfunctions are Dirac delta functions, $\delta(x-x_1)$

* Projecting $\delta(x-x_1)$ onto $\psi$:

* $\int \delta(x-x_1) \psi(x)dx = \psi(x_1)$

* The probability coefficient for every point is then $\psi^*(x)\psi(x)$


In [ ]:
xy_operate(nmax=5, measurements=1000)

interactive(children=(Dropdown(description='nx', options=(1, 2, 3, 4, 5), value=1), Dropdown(description='ny',…

## Momentum operator $\hat{p}_x$

* $\hat{p}_x = -i \hbar \frac{d}{d x}$
* Eigenfunctions: $f_{p_1} = c e^{i p x / \hbar}$
* Eigenvalues: $p_x$

In [38]:
p_measure(10)

interactive(children=(FloatSlider(value=1.0, description='length (nm)', max=2.0, min=0.1), Dropdown(descriptio…